In [1]:
from google.colab import files
uploaded = files.upload()

Saving fer2013.tar.gz to fer2013.tar.gz


In [2]:
import glob, os
tar_files = glob.glob('*.gz')
print('Found:', tar_files)
os.system(f'tar -xzf "{tar_files[0]}"')
!ls fer2013/

Found: ['fer2013.tar.gz']
fer2013.bib  fer2013.csv  README


In [3]:
!pip install wandb -q
import wandb
wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 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


wandb: WARNING Invalid choice


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: mkakh22 (mkakh22-free-university-of-tbilisi-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [4]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import wandb
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
print(f'CUDA available: {torch.cuda.is_available()}')

Using device: cuda
CUDA available: True


In [5]:
df = pd.read_csv('fer2013/fer2013.csv')
print(f'Dataset shape: {df.shape}')
print(df['Usage'].value_counts())
emotion_names = {0:'Angry',1:'Disgust',2:'Fear',3:'Happy',4:'Sad',5:'Surprise',6:'Neutral'}
for k,v in df[df['Usage']=='Training']['emotion'].value_counts().sort_index().items():
    print(f'  {k} ({emotion_names[k]}): {v}')

Dataset shape: (35887, 3)
Usage
Training       28709
PublicTest      3589
PrivateTest     3589
Name: count, dtype: int64
  0 (Angry): 3995
  1 (Disgust): 436
  2 (Fear): 4097
  3 (Happy): 7215
  4 (Sad): 4830
  5 (Surprise): 3171
  6 (Neutral): 4965


In [6]:
def parse_pixels(pixel_string):
    return np.array(pixel_string.split(), dtype=np.float32).reshape(48, 48)

df['pixel_array'] = df['pixels'].apply(parse_pixels)
train_df = df[df['Usage'] == 'Training'].reset_index(drop=True)
val_df   = df[df['Usage'] == 'PublicTest'].reset_index(drop=True)
test_df  = df[df['Usage'] == 'PrivateTest'].reset_index(drop=True)
print(f'Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}')

class FERDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.data = dataframe
        self.transform = transform
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        pixels = self.data['pixel_array'][idx]
        label  = int(self.data['emotion'][idx])
        image  = torch.tensor(pixels / 255.0, dtype=torch.float32).unsqueeze(0)
        if self.transform:
            image = self.transform(image)
        return image, label

train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
])
train_dataset = FERDataset(train_df, transform=train_transform)
val_dataset   = FERDataset(val_df)
test_dataset  = FERDataset(test_df)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_dataset,   batch_size=64, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_dataset,  batch_size=64, shuffle=False, num_workers=2)
print('Dataloaders ready!')

Train: 28709, Val: 3589, Test: 3589
Dataloaders ready!


In [7]:
def train_model(model, train_loader, val_loader, config, run_name):
    wandb.init(
        project='fer2013-experiments',
        entity='mkakh22-free-university-of-tbilisi-',
        name=run_name,
        config=config
    )
    if config['optimizer_name'] == 'adam':
        optimizer = optim.Adam(model.parameters(), lr=config['lr'],
                               weight_decay=config.get('weight_decay', 0))
    else:
        optimizer = optim.SGD(model.parameters(), lr=config['lr'],
                              momentum=0.9, weight_decay=config.get('weight_decay', 0))
    criterion = nn.CrossEntropyLoss()
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)
    model = model.to(device)
    best_val_acc = 0
    for epoch in range(config['epochs']):
        model.train()
        train_loss, train_correct, train_total = 0, 0, 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            train_loss    += loss.item()
            _, predicted   = outputs.max(1)
            train_correct += predicted.eq(labels).sum().item()
            train_total   += labels.size(0)
        model.eval()
        val_loss, val_correct, val_total = 0, 0, 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)
                val_loss    += loss.item()
                _, predicted = outputs.max(1)
                val_correct += predicted.eq(labels).sum().item()
                val_total   += labels.size(0)
        train_acc      = train_correct / train_total
        val_acc        = val_correct   / val_total
        avg_train_loss = train_loss / len(train_loader)
        avg_val_loss   = val_loss   / len(val_loader)
        scheduler.step()
        wandb.log({'epoch': epoch+1, 'train_loss': avg_train_loss,
                   'val_loss': avg_val_loss, 'train_acc': train_acc, 'val_acc': val_acc})
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), f'{run_name}_best.pth')
        if (epoch + 1) % 5 == 0:
            print(f'Epoch {epoch+1}/{config["epochs"]} | '
                  f'Train Loss: {avg_train_loss:.4f} Acc: {train_acc:.4f} | '
                  f'Val Loss: {avg_val_loss:.4f} Acc: {val_acc:.4f}')
    print(f'Best Val Accuracy: {best_val_acc:.4f}')
    wandb.finish()
    return model, best_val_acc

In [8]:
class TinyMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(48 * 48, 64), nn.ReLU(),
            nn.Linear(64, 7)
        )
    def forward(self, x):
        return self.net(x)

model1 = TinyMLP()
print(f'TinyMLP parameters: {sum(p.numel() for p in model1.parameters()):,}')
config1 = {
    'architecture': 'TinyMLP', 'lr': 0.001, 'epochs': 30,
    'optimizer_name': 'adam', 'weight_decay': 0,
    'notes': 'Baseline: intentionally tiny MLP to demonstrate underfitting'
}
model1, acc1 = train_model(model1, train_loader, val_loader, config1, 'run1_tiny_mlp')
print(f'TinyMLP final val acc: {acc1:.4f}')

TinyMLP parameters: 147,975


Epoch 5/30 | Train Loss: 1.6783 Acc: 0.3398 | Val Loss: 1.6538 Acc: 0.3706
Epoch 10/30 | Train Loss: 1.6389 Acc: 0.3572 | Val Loss: 1.6314 Acc: 0.3742
Epoch 15/30 | Train Loss: 1.6120 Acc: 0.3705 | Val Loss: 1.5987 Acc: 0.3814
Epoch 20/30 | Train Loss: 1.6043 Acc: 0.3741 | Val Loss: 1.5960 Acc: 0.3892
Epoch 25/30 | Train Loss: 1.5949 Acc: 0.3815 | Val Loss: 1.5804 Acc: 0.3909
Epoch 30/30 | Train Loss: 1.5876 Acc: 0.3797 | Val Loss: 1.5704 Acc: 0.4004
Best Val Accuracy: 0.4051


epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
train_acc,▁▄▄▅▅▅▅▆▆▆▇▇▇▇▇▇▇▇█▇██████████
train_loss,█▆▅▅▄▄▄▃▃▃▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁
val_acc,▂▃▁▃▄▅▄▄▆▅▇▄▇▆▆▆▆█▇▆▇▇▇▇▇▆▇███
val_loss,█▇█▆▆▅▅▅▃▄▃▄▂▃▃▂▃▂▂▃▂▂▁▂▂▂▁▁▁▁
epoch,30
train_acc,0.37974
train_loss,1.58759
val_acc,0.40039
val_loss,1.57036


TinyMLP final val acc: 0.4051


In [9]:
class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Flatten(),
            nn.Linear(64 * 12 * 12, 256), nn.ReLU(),
            nn.Linear(256, 7)
        )
    def forward(self, x):
        return self.net(x)

model2 = SmallCNN()
print(f'SmallCNN parameters: {sum(p.numel() for p in model2.parameters()):,}')
config2 = {
    'architecture': 'SmallCNN', 'lr': 0.001, 'epochs': 30,
    'optimizer_name': 'adam', 'weight_decay': 0,
    'notes': 'Small CNN - step up from MLP, no regularization yet'
}
model2, acc2 = train_model(model2, train_loader, val_loader, config2, 'run2_small_cnn')
print(f'SmallCNN final val acc: {acc2:.4f}')

SmallCNN parameters: 2,380,167


Epoch 5/30 | Train Loss: 1.3120 Acc: 0.4953 | Val Loss: 1.3085 Acc: 0.4954
Epoch 10/30 | Train Loss: 1.1656 Acc: 0.5578 | Val Loss: 1.2388 Acc: 0.5244
Epoch 15/30 | Train Loss: 1.0576 Acc: 0.6019 | Val Loss: 1.2180 Acc: 0.5414
Epoch 20/30 | Train Loss: 0.9970 Acc: 0.6258 | Val Loss: 1.2172 Acc: 0.5511
Epoch 25/30 | Train Loss: 0.9372 Acc: 0.6523 | Val Loss: 1.2277 Acc: 0.5575
Epoch 30/30 | Train Loss: 0.9091 Acc: 0.6641 | Val Loss: 1.2362 Acc: 0.5634
Best Val Accuracy: 0.5670


epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
train_acc,▁▃▄▄▄▅▅▅▆▆▆▆▆▇▇▇▇▇▇▇▇█████████
train_loss,█▇▆▅▅▄▄▄▄▃▃▃▃▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁
val_acc,▁▂▃▄▅▅▅▆▆▆▆▇▇▇▇▇█▇▇▇██████████
val_loss,█▆▅▃▃▂▂▂▁▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▂▂▂▂
epoch,30
train_acc,0.66408
train_loss,0.90914
val_acc,0.56339
val_loss,1.23619


SmallCNN final val acc: 0.5670


In [10]:
class LargeCNN_NoReg(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 64, 3, padding=1),   nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1),  nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(128, 256, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Flatten(),
            nn.Linear(256 * 6 * 6, 1024), nn.ReLU(),
            nn.Linear(1024, 512), nn.ReLU(),
            nn.Linear(512, 7)
        )
    def forward(self, x):
        return self.net(x)

model3 = LargeCNN_NoReg()
print(f'LargeCNN_NoReg parameters: {sum(p.numel() for p in model3.parameters()):,}')
config3 = {
    'architecture': 'LargeCNN_NoReg', 'lr': 0.001, 'epochs': 50,
    'optimizer_name': 'adam', 'weight_decay': 0,
    'notes': 'Large CNN NO regularization - intentional overfitting demo'
}
model3, acc3 = train_model(model3, train_loader, val_loader, config3, 'run3_large_cnn_overfit')
print(f'LargeCNN_NoReg final val acc: {acc3:.4f}')

LargeCNN_NoReg parameters: 10,336,263


Epoch 5/50 | Train Loss: 1.2091 Acc: 0.5375 | Val Loss: 1.2220 Acc: 0.5230
Epoch 10/50 | Train Loss: 0.9967 Acc: 0.6222 | Val Loss: 1.1624 Acc: 0.5782
Epoch 15/50 | Train Loss: 0.7773 Acc: 0.7116 | Val Loss: 1.2415 Acc: 0.5809
Epoch 20/50 | Train Loss: 0.6262 Acc: 0.7699 | Val Loss: 1.3795 Acc: 0.5957
Epoch 25/50 | Train Loss: 0.4723 Acc: 0.8327 | Val Loss: 1.5837 Acc: 0.5954
Epoch 30/50 | Train Loss: 0.3945 Acc: 0.8575 | Val Loss: 1.7127 Acc: 0.5954
Epoch 35/50 | Train Loss: 0.3218 Acc: 0.8869 | Val Loss: 1.8860 Acc: 0.5943
Epoch 40/50 | Train Loss: 0.2836 Acc: 0.9004 | Val Loss: 1.9833 Acc: 0.5988
Epoch 45/50 | Train Loss: 0.2462 Acc: 0.9130 | Val Loss: 2.1127 Acc: 0.5974
Epoch 50/50 | Train Loss: 0.2248 Acc: 0.9223 | Val Loss: 2.1794 Acc: 0.6002
Best Val Accuracy: 0.6021


epoch,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇███
train_acc,▁▂▃▃▄▄▄▄▅▅▅▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇██████████████
train_loss,█▇▆▆▆▅▅▅▅▄▄▄▄▃▃▃▃▃▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_acc,▁▃▄▅▆▇▇▇█▇█▇▇▇██████▇███████████████████
val_loss,▄▃▂▂▁▁▁▁▁▁▁▂▂▂▂▃▃▃▄▄▄▅▅▅▅▆▆▆▆▆▇▇▇▇▇█████
epoch,50
train_acc,0.92232
train_loss,0.22477
val_acc,0.60017
val_loss,2.17939


LargeCNN_NoReg final val acc: 0.6021


In [12]:
class RegularizedCNN(nn.Module):
    def __init__(self, dropout_rate=0.5):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 64, 3, padding=1),   nn.BatchNorm2d(64),  nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1),  nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 6 * 6, 1024), nn.ReLU(), nn.Dropout(dropout_rate),
            nn.Linear(1024, 512),          nn.ReLU(), nn.Dropout(dropout_rate),
            nn.Linear(512, 7)
        )
    def forward(self, x):
        return self.classifier(self.features(x))

model4 = RegularizedCNN(dropout_rate=0.5)
print(f'RegularizedCNN parameters: {sum(p.numel() for p in model4.parameters()):,}')
config4 = {
    'architecture': 'RegularizedCNN', 'lr': 0.001, 'epochs': 30,
    'optimizer_name': 'adam', 'weight_decay': 1e-4, 'dropout': 0.5,
    'notes': 'BatchNorm + Dropout(0.5) + L2 - best model'
}
model4, acc4 = train_model(model4, train_loader, val_loader, config4, 'run4_regularized_cnn')
print(f'RegularizedCNN final val acc: {acc4:.4f}')

RegularizedCNN parameters: 10,337,159


epoch,▁
train_acc,▁
train_loss,▁
val_acc,▁
val_loss,▁
epoch,1
train_acc,0.24125
train_loss,1.84162
val_acc,0.30566
val_loss,1.71059


Epoch 5/30 | Train Loss: 1.4940 Acc: 0.4002 | Val Loss: 1.4537 Acc: 0.4333
Epoch 10/30 | Train Loss: 1.3831 Acc: 0.4351 | Val Loss: 1.3364 Acc: 0.4625
Epoch 15/30 | Train Loss: 1.2115 Acc: 0.5322 | Val Loss: 1.2333 Acc: 0.5536
Epoch 20/30 | Train Loss: 1.0810 Acc: 0.5894 | Val Loss: 1.1334 Acc: 0.5787
Epoch 25/30 | Train Loss: 0.9606 Acc: 0.6398 | Val Loss: 1.0887 Acc: 0.5862
Epoch 30/30 | Train Loss: 0.8737 Acc: 0.6736 | Val Loss: 1.0296 Acc: 0.6211
Best Val Accuracy: 0.6211


epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
train_acc,▁▂▃▃▄▄▄▄▄▄▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇█████
train_loss,█▇▆▆▅▅▅▅▅▅▄▄▄▄▃▃▃▃▃▂▂▂▂▂▂▁▁▁▁▁
val_acc,▁▁▃▃▄▄▃▄▄▄▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇████
val_loss,██▇▆▆▆▆▅▅▄▄▄▄▃▃▃▃▃▂▂▂▂▂▂▂▂▁▁▁▁
epoch,30
train_acc,0.67359
train_loss,0.87369
val_acc,0.62106
val_loss,1.02962


RegularizedCNN final val acc: 0.6211


In [14]:
lr_results = {}
for lr in [0.01, 0.001, 0.0001]:
    model = RegularizedCNN(dropout_rate=0.5)
    config = {'architecture': 'RegularizedCNN', 'lr': lr, 'epochs': 20,
              'optimizer_name': 'adam', 'weight_decay': 1e-4, 'dropout': 0.5,
              'notes': f'LR sweep: lr={lr}'}
    _, acc = train_model(model, train_loader, val_loader, config, f'run_lr_{lr}')
    lr_results[lr] = acc
    print(f'LR={lr}: val_acc={acc:.4f}')
print('Best LR:', max(lr_results, key=lr_results.get))

Epoch 5/20 | Train Loss: 1.8176 Acc: 0.2483 | Val Loss: 1.8169 Acc: 0.2494
Epoch 10/20 | Train Loss: 1.8104 Acc: 0.2508 | Val Loss: 1.8072 Acc: 0.2530
Epoch 15/20 | Train Loss: 1.7610 Acc: 0.2751 | Val Loss: 1.6869 Acc: 0.3146
Epoch 20/20 | Train Loss: 1.5481 Acc: 0.3925 | Val Loss: 1.4794 Acc: 0.4238
Best Val Accuracy: 0.4238


epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
train_acc,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▂▄▆▇██
train_loss,█▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁
val_acc,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▄▆▇▇▇█
val_loss,█▆▆▆▆▆▆▆▆▆▆▆▆▆▄▃▂▂▂▁
epoch,20
train_acc,0.39246
train_loss,1.54806
val_acc,0.42379
val_loss,1.47942


LR=0.01: val_acc=0.4238


Epoch 5/20 | Train Loss: 1.4389 Acc: 0.4268 | Val Loss: 1.3889 Acc: 0.4723
Epoch 10/20 | Train Loss: 1.3085 Acc: 0.4855 | Val Loss: 1.3332 Acc: 0.5052
Epoch 15/20 | Train Loss: 1.1513 Acc: 0.5588 | Val Loss: 1.1755 Acc: 0.5564
Epoch 20/20 | Train Loss: 1.0526 Acc: 0.6067 | Val Loss: 1.1259 Acc: 0.5751
Best Val Accuracy: 0.5751


epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
train_acc,▁▃▃▄▄▅▅▅▅▆▆▆▇▇▇▇▇███
train_loss,█▆▅▅▅▄▄▄▄▃▃▃▂▂▂▂▂▁▁▁
val_acc,▁▃▄▄▅▅▆▆▇▆▇▇▇███▇▇▇█
val_loss,█▆▅▅▄▄▄▃▃▃▂▂▂▂▂▁▂▂▂▁
epoch,20
train_acc,0.60667
train_loss,1.05259
val_acc,0.57509
val_loss,1.12592


LR=0.001: val_acc=0.5751


Epoch 5/20 | Train Loss: 1.3139 Acc: 0.5005 | Val Loss: 1.2414 Acc: 0.5261
Epoch 10/20 | Train Loss: 1.1869 Acc: 0.5502 | Val Loss: 1.1411 Acc: 0.5595
Epoch 15/20 | Train Loss: 1.0949 Acc: 0.5865 | Val Loss: 1.0884 Acc: 0.5854
Epoch 20/20 | Train Loss: 1.0501 Acc: 0.6034 | Val Loss: 1.0929 Acc: 0.5879
Best Val Accuracy: 0.5901


epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
train_acc,▁▄▅▅▆▆▆▆▇▇▇▇▇███████
train_loss,█▆▅▄▄▃▃▃▃▂▂▂▂▂▁▁▁▁▁▁
val_acc,▁▃▄▄▅▆▆▆▇▇▇▇███▇████
val_loss,█▆▅▅▄▃▃▃▂▂▂▂▂▁▁▂▁▁▁▁
epoch,20
train_acc,0.60343
train_loss,1.0501
val_acc,0.58791
val_loss,1.09289


LR=0.0001: val_acc=0.5901
Best LR: 0.0001


In [16]:
dropout_results = {}
for dropout in [0.3, 0.5, 0.7]:
    model = RegularizedCNN(dropout_rate=dropout)
    config = {'architecture': 'RegularizedCNN', 'lr': 0.001, 'epochs': 20,
              'optimizer_name': 'adam', 'weight_decay': 1e-4, 'dropout': dropout,
              'notes': f'Dropout sweep: dropout={dropout}'}
    _, acc = train_model(model, train_loader, val_loader, config, f'run_dropout_{dropout}')
    dropout_results[dropout] = acc
    print(f'Dropout={dropout}: val_acc={acc:.4f}')
print('Best Dropout:', max(dropout_results, key=dropout_results.get))

epoch,▁
train_acc,▁
train_loss,▁
val_acc,▁
val_loss,▁
epoch,1
train_acc,0.30196
train_loss,1.75291
val_acc,0.3859
val_loss,1.52533


Epoch 5/20 | Train Loss: 1.2552 Acc: 0.5202 | Val Loss: 1.2626 Acc: 0.5235
Epoch 10/20 | Train Loss: 1.1198 Acc: 0.5734 | Val Loss: 1.1588 Acc: 0.5673
Epoch 15/20 | Train Loss: 0.9787 Acc: 0.6280 | Val Loss: 1.0516 Acc: 0.6077
Epoch 20/20 | Train Loss: 0.8949 Acc: 0.6646 | Val Loss: 1.0374 Acc: 0.6124
Best Val Accuracy: 0.6205


epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
train_acc,▁▂▃▃▄▄▄▅▅▅▆▆▇▇▇▇▇███
train_loss,█▇▆▆▅▅▅▄▄▄▃▃▂▂▂▂▂▁▁▁
val_acc,▂▁▁▄▃▃▅▅▃▆▇▇▇█▇█████
val_loss,███▅▆▅▄▃▆▄▂▂▁▁▁▁▁▁▁▁
epoch,20
train_acc,0.66463
train_loss,0.89489
val_acc,0.61243
val_loss,1.0374


In [17]:
opt_results = {}
for opt_name in ['adam', 'sgd']:
    model = RegularizedCNN(dropout_rate=0.5)
    config = {'architecture': 'RegularizedCNN', 'lr': 0.001, 'epochs': 20,
              'optimizer_name': opt_name, 'weight_decay': 1e-4, 'dropout': 0.5,
              'notes': f'Optimizer sweep: {opt_name}'}
    _, acc = train_model(model, train_loader, val_loader, config, f'run_opt_{opt_name}')
    opt_results[opt_name] = acc
    print(f'Optimizer={opt_name}: val_acc={acc:.4f}')
print('Results:', opt_results)

Epoch 5/20 | Train Loss: 1.4426 Acc: 0.4324 | Val Loss: 1.4091 Acc: 0.4712
Epoch 10/20 | Train Loss: 1.3092 Acc: 0.4868 | Val Loss: 1.3305 Acc: 0.4918
Epoch 15/20 | Train Loss: 1.1321 Acc: 0.5724 | Val Loss: 1.1670 Acc: 0.5645
Epoch 20/20 | Train Loss: 1.0326 Acc: 0.6131 | Val Loss: 1.1184 Acc: 0.5907
Best Val Accuracy: 0.5907


epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
train_acc,▁▃▄▄▅▅▅▅▅▆▆▆▇▇▇▇████
train_loss,█▆▅▅▅▄▄▄▄▃▃▃▂▂▂▂▁▁▁▁
val_acc,▁▄▄▄▅▆▆▆▆▆▇▇▇▇▇█████
val_loss,█▆▅▅▄▄▄▄▄▃▃▂▂▂▂▂▁▁▁▁
epoch,20
train_acc,0.61308
train_loss,1.03263
val_acc,0.59069
val_loss,1.11839


Optimizer=adam: val_acc=0.5907


Epoch 5/20 | Train Loss: 1.4018 Acc: 0.4640 | Val Loss: 1.3364 Acc: 0.4935
Epoch 10/20 | Train Loss: 1.2668 Acc: 0.5149 | Val Loss: 1.2804 Acc: 0.5146
Epoch 15/20 | Train Loss: 1.1751 Acc: 0.5559 | Val Loss: 1.1568 Acc: 0.5506
Epoch 20/20 | Train Loss: 1.1274 Acc: 0.5755 | Val Loss: 1.1201 Acc: 0.5795
Best Val Accuracy: 0.5795


epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
train_acc,▁▃▄▅▅▆▆▆▇▇▇▇▇▇██████
train_loss,█▆▅▅▄▄▃▃▃▃▂▂▂▂▂▁▁▁▁▁
val_acc,▁▃▄▅▅▄▆▆▇▆▇▇▇▇▇▇▇▇▇█
val_loss,█▆▅▄▄▅▃▃▂▃▂▂▂▂▂▁▁▁▁▁
epoch,20
train_acc,0.57546
train_loss,1.12742
val_acc,0.57955
val_loss,1.12014


Optimizer=sgd: val_acc=0.5795
Results: {'adam': 0.5906937865700752, 'sgd': 0.5795486207857342}


In [18]:
print('=' * 60)
print('FINAL RESULTS SUMMARY')
print('=' * 60)
print(f'1. TinyMLP          val_acc={acc1:.4f}  -> UNDERFITTING')
print(f'2. SmallCNN         val_acc={acc2:.4f}  -> Baseline')
print(f'3. LargeCNN_NoReg   val_acc={acc3:.4f}  -> OVERFITTING')
print(f'4. RegularizedCNN   val_acc={acc4:.4f}  -> BEST MODEL')
print()
print('Key findings:')
print('- MLP underfits: spatial structure lost when flattening')
print('- LargeCNN overfits: train_acc >> val_acc without regularization')
print('- RegularizedCNN: BatchNorm+Dropout closes the train/val gap')

FINAL RESULTS SUMMARY
1. TinyMLP          val_acc=0.4051  -> UNDERFITTING
2. SmallCNN         val_acc=0.5670  -> Baseline
3. LargeCNN_NoReg   val_acc=0.6021  -> OVERFITTING
4. RegularizedCNN   val_acc=0.6211  -> BEST MODEL

Key findings:
- MLP underfits: spatial structure lost when flattening
- LargeCNN overfits: train_acc >> val_acc without regularization
- RegularizedCNN: BatchNorm+Dropout closes the train/val gap
